# Test cases for SPH implementation in Peano


In this notebook, we set up and run a battery of well-established tests for SPH codes.

This includes:

- 1. Sedov-Taylor Blast (current): 

    Particles are initialised at rest in regular grid configuration and periodic BC. Gas has negligible presure. Energy is injected to a (or group of) central particle and the blast should propagate isotropically, up to boundary effects. See, e.g., Springel and Hernquist, Mon. Not. R. Astron. Soc. 333, 649–664 (2002).
    

- 2. Sod shockwave
- 3. Gresho-Chan vortex
- 4. Noh Problem
- 5. Square test
- 6. Kelvin-Helmholtz Instability
- (any other relevant...)

In [ ]:
import os

import peano4
import peano4.dastgen2
import peano4.toolbox
import peano4.toolbox.particles
import dastgen2

In [ ]:
# If we want to start a project from scratch, we may want to remove some of the generated code
!make clean; rm -r actions/ vertexdata/ repositories/ globaldata/ celldata/ observers/ 

project = peano4.Project( ["examples", "particles"], "sph-peano-tests", "." )

## Prepare code

Peano's API does not know which settings to use on the present system. To make it copy/clone the settings identified by ./configure, we ask it to parse the generated configuration scripts. 

In [ ]:
build_mode = peano4.output.CompileMode.Asserts
#build_mode = peano4.output.CompileMode.Release
project.output.makefile.parse_configure_script_outcome( "../../.." )
project.output.makefile.set_mode( build_mode )

## Model the particle

We create the particle through the toolbox. A particle by definition has a position, so we just need to add everything else for SPH:
- Velocity array
- Mass
- Density ($\rho$)
- Pressure ($P$)
- Internal Energy ($u$)
- Kernel smoothing length ($h$)

We add a marker to indicate the next calculation step required for a particle so we don't perform the same step on a particle twice in a row.

We also add an array to store the new position of a particle after a timestep is completed. We will use this to move the particle at the start of the subsequent timestep.

In [ ]:
particle = peano4.toolbox.particles.Particle("Particle")

# Define properties
v_attr = peano4.dastgen2.Peano4DoubleArray("v","Dimensions")
mass_attr    = dastgen2.attributes.Double("mass")
density_attr = dastgen2.attributes.Double("density")
pressure_attr= dastgen2.attributes.Double("pressure")
u_attr       = dastgen2.attributes.Double("u")
smoothL_attr = dastgen2.attributes.Double("smoothingLength")

# For variable smoothing length
has_converged_attr= dastgen2.attributes.Boolean("hasConverged")
smoothL_prev_attr = dastgen2.attributes.Double("smoothingLengthPrevious")
Omega_attr        = dastgen2.attributes.Double("Omega")
rhoDot_attr       = dastgen2.attributes.Double("rhoDot")
DebugNeg_attr     = dastgen2.attributes.Double("DebugNeg")

#newX_attr    = peano4.dastgen2.Peano4DoubleArray("newX", "Dimensions") # updated copy of X

# Add particle properties
particle.data.add_attribute(v_attr)
particle.data.add_attribute(mass_attr)
particle.data.add_attribute(density_attr)
particle.data.add_attribute(pressure_attr)
particle.data.add_attribute(u_attr)
particle.data.add_attribute(smoothL_attr)
# For variable smoothing length
particle.data.add_attribute(has_converged_attr)
particle.data.add_attribute(smoothL_prev_attr)
particle.data.add_attribute(Omega_attr)
particle.data.add_attribute(rhoDot_attr)
particle.data.add_attribute(DebugNeg_attr)

# Check these
#particle.data.add_attribute(newX_attr)

particles = peano4.toolbox.particles.ParticleSet(particle)

This particle has to be added to the project. This way the project knows that we are working with this guy. We associate the particle with the vertices, as we work with the pidt (particle in dual tree) algorithm. 

In [ ]:
project.datamodel.add_global_object(particle)
project.datamodel.add_vertex       (particles)

## Export global SPH constants

There are plenty of constant simulation paramaters to specify for our SPH simulation - we do that here. 

In [ ]:
Dimensions = 2
project.output.makefile.set_dimension(Dimensions)

In [ ]:
from numpy import pi

# Estimate free parameter for smoothing length based on how many average ngbs we want to use
N_ngbs_mean = 10.0 # 57.9

# SPH parameters
GAMMA = str(5/3)
#DYNAMIC_VISCOSITY     = "0.00001"
KERNEL_SUPPORT_RAD = 2  # Cubic Spline = 2. Can be different in other kernels.

if(Dimensions==2):
    SMOOTH_L_FAC = str( (1.   /pi*N_ngbs_mean)**(1./2.)/KERNEL_SUPPORT_RAD ) 
    CUBIC_SPLINE_SIGMA = str(10/(7*pi))
    
elif(Dimensions==3):
    SMOOTH_L_FAC = str( (3./4./pi*N_ngbs_mean)**(1./3.)/KERNEL_SUPPORT_RAD ) 
    CUBIC_SPLINE_SIGMA = str(1/pi)

print(SMOOTH_L_FAC, CUBIC_SPLINE_SIGMA)

# parameters specific to a given problem (e.g., Sedov blast, Sod wave, ...)
ENERGY_SPIKE = "1.0"
BLAST_CENTER = "{0.5, 0.5}"
#SPIKE_RADIUS = "0.08"

project.constants.export_const_with_type("GAMMA", GAMMA, "double")
project.constants.export_const_with_type("SMOOTH_L_FAC"      , SMOOTH_L_FAC      , "double")
project.constants.export_const_with_type("CUBIC_SPLINE_SIGMA", CUBIC_SPLINE_SIGMA, "double")
project.constants.export_const_with_type("ENERGY_SPIKE", ENERGY_SPIKE, "double")
project.constants.export_const_with_type("BLAST_CENTER", BLAST_CENTER, "auto")
#project.constants.export_const_with_type("SPIKE_RADIUS", SPIKE_RADIUS, "double")
#project.constants.export_const_with_type("DYNAMIC_VISCOSITY", DYNAMIC_VISCOSITY, "double")

## Model the setup phases

## Initialize regular grid

In [ ]:
mesh_size = 0.2

particle_tree_analysis = peano4.toolbox.particles.ParticleTreeAnalysis(particles)
project.datamodel.add_cell(particle_tree_analysis.cell_marker)   # read docu of ParticleTreeAnalysis

create_grid = peano4.solversteps.Step( name="CreateGrid", add_user_defined_actions=False )
create_grid.add_action_set( peano4.toolbox.CreateRegularGrid(mesh_size) )
create_grid.use_vertex(particles)
create_grid.use_cell(particle_tree_analysis.cell_marker)
project.solversteps.add_step(create_grid)

## Initial conditions for particles

In [ ]:
initialisation_snippet = """
 
 // Internal cutoff radious used by Peano. Should always be larger than actual SPH cutoff radious (2h).
 particle->setCutOffRadius(0.1);
 
 // Initial Conditions for the particular SPH test
 //
 // 1. Sedov Blast: particles are initialised at rest in regular grid configuration and periodic BC. 
 // Energy is injected to a (or group of) central particle and the blast should propagate
 // isotropically, up to boundary effects.
 // See, e.g., Springel and Hernquist, Mon. Not. R. Astron. Soc. 333, 649–664 (2002).
 //
 
 particle->setV( 0, 0 );
 particle->setV( 1, 0 );
 
 particle->setDensity ( 1.0 );        
 particle->setMass    ( particle->getDensity()/10 );       
 particle->setPressure( 1e-7);

 tarch::la::Vector<Dimensions,double> blast_cent(0.0);
 const double dist_blast = tarch::la::norm2( particle->getX() - blast_cent );

// if ( tarch::la::smaller( dist_blast, 0.3 ) ) {
   particle->setU (0.1);
// } 
 
 particle->setSmoothingLength( SMOOTH_L_FAC * std::pow(particle->getMass()/particle->getDensity(), 1.0/Dimensions) );
 particle->setSmoothingLengthPrevious(1E-1);
 particle->setOmega (0.0);        
 particle->setRhoDot(0.0);        
 particle->setDebugNeg(-1.0);        

"""

init_setup = peano4.solversteps.Step( name="Init", add_user_defined_actions=False )
init_setup.use_vertex(particles)
init_setup.use_cell(particle_tree_analysis.cell_marker)
init_setup.add_action_set( peano4.toolbox.particles.UpdateParticleGridAssociation(particles) )
init_setup.add_action_set( peano4.toolbox.particles.InsertRandomParticlesIntoUnrefinedCells(
  particle_set=particles,
  average_distance_between_particles=mesh_size/2,
  initialisation_call=initialisation_snippet,
  noise=False,
  additional_includes=""" 
    #include "Constants.h" 
    """ ))
init_setup.add_action_set( particle_tree_analysis )
project.solversteps.add_step(init_setup)

In [ ]:
1E0/(pi*(0.2/10/2)**2)

## Particle behaviour

We stick to exactly the same particle behaviour from a code point of view. However, we use a pre-manufactured toolbox  into which we inject the actual code. The important fact here is that we now do not simply rely on the particles of the adjacent vertices of a cell. Instead, we work with active and local particle sets.

In [ ]:
from IPython.display import Image
!ls ../../python/peano4/toolbox/particles
Image("../../../python/peano4/toolbox/particles/dependency_sets.png")

To understand what the individual particles in the sketch above are supposed to mean and how they are maintained and exposed to the user, please consult the documentation of the ParticleParticleInteraction class in the Python toolbox directory. It contains a lot explanations.

## Density loop with variable smoothing length

In the case of variable smoothing length, $h\neq{\rm const.}$, we need to calculate this alongside with density in a self-consistent way. We proceed in two steps as follows:

1. We loop over neighbours ($b$) with the current value of $h$, to calculate the various sums:

\begin{align}
\rho_a& = \sum_b m_b W(|{\bf r}_a - {\bf r}_b|, h_a)\\
\delta\Omega_a\equiv\Omega_a -1 &= - \frac{\partial h_a}{\partial \rho_a}\sum_b m_b\frac{\partial W_{ab}(h_a)}{\partial h_a} \qquad\qquad (\delta\Omega_a=0 \quad \textrm{for constant $h$})\\
\frac{{\rm d}{\rho_a}}{{\rm d}t} &= \frac{1}{\Omega_a}\sum_b m_b{v}_{ab}\cdot{\bf \nabla}_aW_{ab}(h_a)
\end{align}
The latter is later used to calculate the initial guess for the Newton-Raphson, which can reduce the number of required iterations.

2. Once the above have been calculated, we solve iteratively for $h$ using Newton-Raphson:

\begin{align}
\zeta(h_a)&\equiv m_a\left(\frac{\eta}{h_a}\right)^{3}-\sum_b m_b W_{ab}(h_a)=0\\
\textrm{Initial guess:} &\qquad\qquad h^{\rm ini}_a =h_a + \Delta t \frac{\partial h_a}{\partial\rho_a}\frac{d\rho_a}{dt}\\ 
\textrm{N-R iteration:}&\qquad\qquad h_{a,{\rm new}}=h_a - \frac{\zeta({h_a})}{\zeta'(h_a)} \qquad \textrm{where:}\qquad \zeta'(h_a)=-\frac{3\rho_a \Omega_a}{h_a}
\end{align}


Once $\rho$ (and $h$) has been calculated, we can evaluate the fluid pressure at the particle's position using the Equation of State:

\begin{align}
P &=(\gamma-1)\rho u \qquad \textrm{Equation of State (EoS)}
\end{align}
Notice that, if $u$ has evolved, we should evaluate $P$ later.

In [ ]:
calc_densities_var_h = peano4.solversteps.Step( "CalcDensitiesAndSmoothingLengthNR" )
calc_densities_var_h.use_vertex(particles)
calc_densities_var_h.use_cell(particle_tree_analysis.cell_marker)
calc_densities_var_h.add_action_set( peano4.toolbox.particles.UpdateParticleGridAssociation(particles) )

particle_particle_compute_kernel = """
// The used action set does the run over the local set and the active set. So we
// simply have to feed in the particle-particle interaction.

// Here we calculate the density, Omega and drho/dt sums.
// These are later used in the Newton-Raphson iteration.
    
  tarch::la::Vector<Dimensions,double> distVec = localParticle->getX() - activeParticle->getX();
  const double dist = tarch::la::norm2( localParticle->getX() - activeParticle->getX() );
  
  const double h = localParticle->getSmoothingLength();
  const double cutoffR = 2*h;
  
  if ( tarch::la::greater( dist,0.0 ) ) {
    if ( dist < cutoffR ) {
     
      const double q = dist / h;
      
      // Default SPH kernel (cubic spline)
      double W = CUBIC_SPLINE_SIGMA / std::pow( h, Dimensions );
      
      if (q >= 2) {
        W *= 0.0;
      } else if (q >= 1) {
        W *= (0.25 * (2 - q)*(2 - q)*(2 - q));
      } else {
        W *= (1 - 1.5*q*q*(1 - q/2));
      }
    
      double densityNgbs = activeParticle->getMass() * W;
      localParticle->setDensity( localParticle-> getDensity() + densityNgbs );
      
      // Calculation of dW/dh for Omega
      double dWdh = CUBIC_SPLINE_SIGMA / std::pow( h, Dimensions ); 
      
      if (q >= 2) {
        dWdh *= 0.0;
      } else if (q >= 1) {
        dWdh *= q* (-0.75 * ((2 - q)*(2 - q)));
      } else {
        dWdh *= q* (0.75*q * (3*q - 4));
      }
      
      dWdh += Dimensions*W;
      dWdh *= -1.0 / h;
      
      double dhOverdrho = - h/localParticle->getDensity()/Dimensions;
         
      localParticle->setOmega( localParticle-> getOmega() + dhOverdrho * activeParticle->getMass() * dWdh );

//      assertion3( localParticle->getOmega() <= 1.0 , localParticle->toString(), localParticle->getOmega(), dWdh );
      assertion3( not std::isnan( localParticle->getOmega() ), localParticle->toString(), localParticle->getOmega(), dWdh );
      
      // Calculation of dRho/dt.
      // Used to predict smoothing length for initial guess of N-R iterations.
      
      double delW = CUBIC_SPLINE_SIGMA / std::pow( h, Dimensions );

      if (q >= 2) {
        delW *= 0;
      } else if (q >= 1) {
        delW *= (-0.75 * ((2 - q)*(2 - q)));
      } else {
        delW *= (0.75*q * (3*q - 4));
      }
  
      tarch::la::Vector<Dimensions,double> gradW = delW*distVec/dist/h;
  
      localParticle->setRhoDot( localParticle-> getRhoDot() + 
        activeParticle->getMass() * tarch::la::dot( (localParticle->getV() - activeParticle->getV()), gradW )
      );
           
    }
  
  }
    
"""

touch_particle_last_time_compute_kernel = """
  // After the loop over ngbs (and with this the density sum) is done, we can do the Newton-Raphson iteration:
  // Notice that each iteration requires to do the full density loop.
  
  localParticle->setOmega ( 1.0 - localParticle->getOmega() );                       // Final value of Omega
//  localParticle->setRhoDot( localParticle->getRhoDot() / localParticle->getOmega() );  // Final value of rhoDot

  // Correct the value of Omega 
//  if(localParticle->getOmega() < 0){ (PySPH)
  if(localParticle->getOmega() <= 0){
    localParticle->setOmega(1.0);
  }

  assertion2( not localParticle->getOmega()==0, localParticle->toString(), localParticle->getSmoothingLength() );

  const double h = localParticle->getSmoothingLength(); // value at the start of NR
  const double dhOverdrho = - h/localParticle->getDensity()/Dimensions; 
    
//  const double hIni = h + 0.001 * dhOverdrho * localParticle->getRhoDot();  // Initial guess
  const double hIni = h;
//  const double hOld = localParticle->getSmoothingLengthPrevious();
  
  const double htolerance = 1E-2; // Tolerance for convergence of N-R iterations
  
  int icount = 0;
  int maxiter= 3;

  while( not localParticle->getHasConverged() ){
    // The equation that NR solves is f = rho - rho_sum = 0. Same convention as PySPH, opposite of Price et al.
    double func = localParticle->getMass() * std::pow( SMOOTH_L_FAC/localParticle->getSmoothingLength(), Dimensions );
    func += -localParticle->getDensity();

    double dfuncOverdh = localParticle->getOmega()/dhOverdrho; // insert (-) if following Price et al convention.
    
    // Update value of smoothing length by one N-R iteration
    localParticle->setSmoothingLength( localParticle->getSmoothingLength() - func/dfuncOverdh  );
    
    // Control changes in h
    if(localParticle->getSmoothingLength() > 1.2 * h) {
      localParticle->setSmoothingLength( 1.2*h );
    } else if (localParticle->getSmoothingLength() < 0.8 * h){
      localParticle->setSmoothingLength( 0.8*h );
    }
    
    // Correct h if it has become too small
    if(localParticle->getSmoothingLength()<= 1E-4) {
      localParticle->setSmoothingLength(h);
    }

    // Check if convergence has been achieved
    const double residual = std::abs(localParticle->getSmoothingLength() - h)/hIni > htolerance;
    
    if(residual <= htolerance){
      localParticle->setHasConverged(true);  
    }
    
    icount+=1;
    if(icount>maxiter) break;
    
    // DEBUG checks
    assertion4( not std::isnan(localParticle->getSmoothingLength()), localParticle->toString(), localParticle->getRhoDot(), localParticle->getOmega(),dhOverdrho );
    assertion4( localParticle->getDebugNeg()==-1.0, localParticle->toString(), localParticle->getDebugNeg(), localParticle->getRhoDot(), localParticle->getOmega());
    assertion6( localParticle->getSmoothingLength()>1E-5, localParticle->toString(), localParticle->getSmoothingLength(), func, dfuncOverdh, localParticle->getOmega(),residual );
    
  }
  
  // Finally, we update the pressure with the EoS since density is done by now.
  double eosPressure = localParticle->getDensity() * localParticle->getU() * (GAMMA - 1);
  localParticle->setPressure( (eosPressure < 0) ? 0 : eosPressure );
"""

touch_particle_first_time_compute_kernel = """
  localParticle->setSmoothingLengthPrevious( localParticle->getSmoothingLength() ); // Store old value
  localParticle->setHasConverged(false);
  
  const double h = localParticle->getSmoothingLength();
  
   // Default SPH kernel (cubic spline)
  double W = CUBIC_SPLINE_SIGMA / std::pow( h, Dimensions );
    
  double dhOverdrho = - h/localParticle->getDensity()/Dimensions;                                                                                          
  
  // we'll accumulate the data throughout mesh/particle traversal
  localParticle->setDensity ( localParticle->getMass() * W );                                 // particle's own contribution
  localParticle->setOmega   ( dhOverdrho * localParticle->getMass() * -Dimensions * W / h );  // particle's own contribution (q=0)  
  localParticle->setRhoDot  (0.0);   
  localParticle->setPressure(0.0);  
"""

calc_densities_var_h.add_action_set( peano4.toolbox.particles.ParticleParticleInteraction(
  particle_set=particles,
  particle_particle_interaction_kernel=particle_particle_compute_kernel,
  touch_vertex_first_time_compute_particle_update_kernel=touch_particle_first_time_compute_kernel,
  touch_vertex_last_time_compute_particle_update_kernel=touch_particle_last_time_compute_kernel,
  additional_includes='#include "Constants.h" '
))
calc_densities_var_h.add_action_set( peano4.toolbox.particles.ParticleAMR(particles,particle_tree_analysis,min_particles_per_cell=10) )
calc_densities_var_h.add_action_set( particle_tree_analysis )
project.solversteps.add_step(calc_densities_var_h)

## Internal energy loop

In this step, we calculate $u$ from the first-order ODE:

\begin{align}
\frac{{\rm d}u_a}{{\rm d}t} &=\frac{P_a}{\rho_a^2\Omega_a}\sum_b m_b{\bf v}_{ab}\cdot{\bf \nabla}_aW_{ab}(h_a) + \Lambda_{\rm shock}\\
\end{align}
which we integrate explicitly, i.e. we simply evaluate the terms in the r.h.s. from the previous time step.

In this,

\begin{align}
\Lambda_{\rm shock} \equiv& -\frac{1}{\Omega_a\rho_a}\sum_b m_b v_{{\rm sig},a}\frac{({\bf v}_{ab}\cdot\hat{\bf r}_{ab})^2}{2}F_{ab}(h_a)\\
&+\sum_b m_b\alpha_u v^u_{\rm sig}(u_a - u_b)\frac{1}{2}\left[\frac{F_{ab}(h_a)}{\Omega_a\rho_a}+\frac{F_{ab}(h_b)}{\Omega_b\rho_b}\right]
\end{align}
represent the changes in internal energy due to shocks.

Once $u$ (and $\rho$) has been updated, we can evaluate the pressure from the EoS:

\begin{align}
P &=(\gamma-1)\rho u \qquad \textrm{Equation of State (EoS)}
\end{align}

In [ ]:
calc_int_energies = peano4.solversteps.Step( "CalcInternalEnergies" )
calc_int_energies.use_vertex(particles)
calc_int_energies.use_cell(particle_tree_analysis.cell_marker)
calc_int_energies.add_action_set( peano4.toolbox.particles.UpdateParticleGridAssociation(particles) )
calc_int_energies.add_action_set( peano4.toolbox.particles.ParticleParticleInteraction(
  particle_set=particles,
  particle_particle_interaction_kernel="""
  
  tarch::la::Vector<Dimensions,double> distVec = localParticle->getX() - activeParticle->getX();
  const double dist = tarch::la::norm2(distVec);
  
  const double h = localParticle->getSmoothingLength();
  const double cutoffR = 2*h;
  
  if ( tarch::la::greater( dist,0.0 ) ) {
    if ( dist < cutoffR ) {
      const double q = dist / h;

      const double massNgbs = activeParticle->getMass();
      const double POverRhoSquared     = localParticle ->getPressure()/(localParticle ->getDensity() * localParticle ->getDensity());
      const double POverRhoSquaredNgbs = activeParticle->getPressure()/(activeParticle->getDensity() * activeParticle->getDensity());

      // Default SPH kernel (cubic spline)
      double delW = CUBIC_SPLINE_SIGMA / std::pow( h, Dimensions ); 
    
      if (q >= 2) {
        delW *= 0;
      } else if (q >= 1) {
        delW *= (-0.75 * ((2 - q)*(2 - q)));
      } else {
        delW *= (0.75*q * (3*q - 4));
      }

      tarch::la::Vector<Dimensions,double> gradW = delW*distVec/dist/h;
           
      // Add contribution due to pressure gradients
      double u = -POverRhoSquared/localParticle->getOmega() * massNgbs * tarch::la::dot( (localParticle->getV() - activeParticle->getV()), gradW );

      // Add heating/cooling due to shocks
      const double alphaU = 1;
      const double oneOverOmegaDensity     = 1/( localParticle ->getDensity()*localParticle ->getDensity() );
      const double oneOverOmegaDensityNgbs = 1/( activeParticle->getDensity()*activeParticle->getDensity() );

      double veldotdist = tarch::la::dot( (localParticle->getV() - activeParticle->getV()), distVec/dist );
      double velSigU = std::pow( std::abs( localParticle->getPressure() - activeParticle->getPressure() ) / ( ( localParticle->getDensity() + activeParticle->getDensity() )/2 ) , 0.5 ); // NO GRAVITY
//      double velSigU = std::abs( veldotdist ); // WITH GRAVITY

      // Contribution from Grad_a{W_ab(h_b)}, i.e., same as above but using h_Nbgs.

      const double hNgbs = activeParticle->getSmoothingLength();
      const double qNgbs = dist / hNgbs;
      double delWNgbs = CUBIC_SPLINE_SIGMA / std::pow( hNgbs, Dimensions )/hNgbs;

      if (qNgbs >= 2) {
        delWNgbs *= 0;
      } else if (qNgbs >= 1) {
        delWNgbs *= (-0.75 * ((2 - q)*(2 - q)));
      } else {
        delWNgbs *= (0.75*q * (3*q - 4));
      }

//      tarch::la::Vector<Dimensions,double> gradWNgbs = delWNgbs*distVec/dist/hNgbs;

      double LambdaShock = - 0.5 * oneOverOmegaDensity * massNgbs *velSigU * veldotdist * veldotdist * (delW/h);
      LambdaShock += 0.5*massNgbs * alphaU * velSigU * ( localParticle->getU() - activeParticle->getU() ) * ( (delW/h)*oneOverOmegaDensity + (delWNgbs/hNgbs)*oneOverOmegaDensityNgbs );
      // Note: we need to add extra an term if magnetic fields are present                                                                                                                    
                                                                                                                                                                                              
      u += LambdaShock;                                                                                                                                                                       
                                                                                                                                                                                              
      assertion7( not std::isnan(u), localParticle->toString(), u, activeParticle->getDensity(), POverRhoSquared, POverRhoSquaredNgbs,localParticle->getOmega(), LambdaShock );               
                                                                                                                                                                                              
      // Update specific internal energy. 
      // This satisfies an evolution equation. Here we simply use an explicit integration (forward Euler).
      localParticle->setU( localParticle-> getU() + 0.0001 * u );
    }
  }
  
""",
  touch_vertex_first_time_compute_particle_update_kernel="""

""",
  touch_vertex_last_time_compute_particle_update_kernel=""" 

""",
  additional_includes='#include "Constants.h" '
  ))
calc_int_energies.add_action_set( peano4.toolbox.particles.ParticleAMR(particles,particle_tree_analysis,min_particles_per_cell=10) )
calc_int_energies.add_action_set( particle_tree_analysis )
project.solversteps.add_step(calc_int_energies)

## Move step (force loop)

We move particles to their new positions and updated their velocities. For the latter, each particle has its acceleration calculated using the densities, pressures and velocities of its neighbours according to the Euler equation

\begin{align}
\frac{{\rm d}{\bf v}^a}{{\rm d}t} &=-\sum_b m_b\left[\frac{P_a + q^a_{ab}}{\rho_a^2\Omega_a}{\bf \nabla}_aW_{ab}(h_a) + \frac{P_b + q^b_{ab}}{\rho_b^2\Omega_b}{\bf \nabla}_aW_{ab}(h_b) \right] + {\bf a}^{\rm self-grav}_a + {\bf a}^{\rm others}_a\\
\end{align}

which we integrate explicitly. Here, $q^a_{ab}$ and $q^b_{ab}$ represent the artificial viscosity terms. At the moment, we assume ${\bf a}^{\rm others}={\bf 0}$.

In [ ]:
#move_particles = peano4.solversteps.Step( "Move", add_user_defined_actions=False )
move_particles = peano4.solversteps.Step( "Move" )
move_particles.use_vertex(particles)
move_particles.use_cell(particle_tree_analysis.cell_marker)
move_particles.add_action_set( peano4.toolbox.particles.UpdateParticleGridAssociation(particles) )
move_particles.add_action_set( peano4.toolbox.particles.ParticleParticleInteraction(
  particle_set=particles,
  particle_particle_interaction_kernel="""
  
  tarch::la::Vector<Dimensions,double> distVec = localParticle->getX() - activeParticle->getX();
  tarch::la::Vector<Dimensions,double> accelHydro(0.0);
  const double dist = tarch::la::norm2(distVec);
  
  const double h = localParticle->getSmoothingLength();
  const double cutoffR = 2*h;

  // Hydrodynamics:
  // Here we calculate the acceleration due to pressure forces according to the Euler Equation.
  
  if ( tarch::la::greater( dist,0.0 ) ) {
    if ( dist < cutoffR ) {
    
      const double q = dist / h;

      const double massNgbs = activeParticle->getMass();
      double POverRhoSquared     = localParticle ->getPressure()/(localParticle ->getDensity() * localParticle ->getDensity());
      double POverRhoSquaredNgbs = activeParticle->getPressure()/(activeParticle->getDensity() * activeParticle->getDensity());

      // Default SPH kernel (cubic spline)
      double delW = CUBIC_SPLINE_SIGMA / std::pow( h, Dimensions );

      if (q >= 2) {
        delW *= 0;
      } else if (q >= 1) {
        delW *= (-0.75 * ((2 - q)*(2 - q)));
      } else {
        delW *= (0.75*q * (3*q - 4));
      }

      tarch::la::Vector<Dimensions,double> gradW = delW*distVec/dist/h;
      
      // Contribution from Grad_a{W_ab(h_b)}, i.e., same as above but using h_Nbgs.
      
      const double hNgbs = activeParticle->getSmoothingLength();
      const double qNgbs = dist / hNgbs;
      double delWNgbs = CUBIC_SPLINE_SIGMA / std::pow( hNgbs, Dimensions )/hNgbs;

      if (qNgbs >= 2) {
        delWNgbs *= 0;
      } else if (qNgbs >= 1) {
        delWNgbs *= (-0.75 * ((2 - q)*(2 - q)));
      } else {
        delWNgbs *= (0.75*q * (3*q - 4));
      }
    
      tarch::la::Vector<Dimensions,double> gradWNgbs = delWNgbs*distVec/dist/hNgbs;
      
      // Add Artificial Viscosity (AV) terms
      const double alphaAV = 1;
      const double betaAV  = 2;
      double cSpeed     =  std::pow( GAMMA * localParticle ->getPressure()/localParticle ->getDensity() , 0.5 );          // sound speed
      double cSpeedNgbs =  std::pow( GAMMA * activeParticle->getPressure()/activeParticle->getDensity() , 0.5 );

      double veldotdist = tarch::la::dot( (localParticle->getV() - activeParticle->getV()), distVec/dist );

      assertion6(  std::isfinite(veldotdist)     , localParticle->toString(), veldotdist, localParticle->getV(), activeParticle->getV(), distVec, dist );
      assertion6( not std::abs  (veldotdist)<1e50, localParticle->toString(), veldotdist, localParticle->getV(), activeParticle->getV(), distVec, dist );

      const double vSig     = alphaAV * cSpeed     + betaAV * std::abs(veldotdist);
      const double vSigNgbs = alphaAV * cSpeedNgbs + betaAV * std::abs(veldotdist);

      double qAV = 0;
      double qAVNgbs = 0;
      if(veldotdist < 0){
        qAV     = - 0.5*vSig     * localParticle ->getDensity() * veldotdist;
        qAVNgbs = - 0.5*vSigNgbs * activeParticle->getDensity() * veldotdist;
      }
      
      assertion9( not tarch::la::greater(std::abs(localParticle ->getV(0)),1e+10), localParticle->toString(), accelHydro, activeParticle->getDensity(), POverRhoSquared, POverRhoSquaredNgbs, dist, veldotdist, qAV,qAVNgbs );
      assertion9( not tarch::la::greater(std::abs(localParticle ->getV(1)),1e+10), localParticle->toString(), accelHydro, activeParticle->getDensity(), POverRhoSquared, POverRhoSquaredNgbs, dist, veldotdist, qAV,qAVNgbs );
      assertion9( std::isfinite(localParticle->getV(0)),      localParticle->toString(), accelHydro, activeParticle->getDensity(), POverRhoSquared, POverRhoSquaredNgbs, dist, veldotdist, qAV,qAVNgbs );
      assertion9( std::isfinite(localParticle->getV(1)),      localParticle->toString(), accelHydro, activeParticle->getDensity(), POverRhoSquared, POverRhoSquaredNgbs, dist, veldotdist, qAV,qAVNgbs );
      assertion9( not std::abs(activeParticle->getV(0))<1e+10, localParticle->toString(), accelHydro, activeParticle->getDensity(), POverRhoSquared, POverRhoSquaredNgbs, dist, veldotdist, qAV,qAVNgbs );
      assertion9( not std::abs(activeParticle->getV(1))<1e+10, localParticle->toString(), accelHydro, activeParticle->getDensity(), POverRhoSquared, POverRhoSquaredNgbs, dist, veldotdist, qAV,qAVNgbs );
      assertion9( std::isfinite(qAV    ), localParticle->toString(), activeParticle->getDensity(), POverRhoSquared, POverRhoSquaredNgbs, dist, distVec, veldotdist, localParticle->getV(), activeParticle->getV());
      assertion9( std::isfinite(qAVNgbs), localParticle->toString(), activeParticle->getDensity(), POverRhoSquared, POverRhoSquaredNgbs, dist, distVec, veldotdist, localParticle->getV(), activeParticle->getV());

      POverRhoSquared     += qAV     / (localParticle ->getDensity() * localParticle ->getDensity());
      POverRhoSquaredNgbs += qAVNgbs / (activeParticle->getDensity() * activeParticle->getDensity());

      // Update velocity
      accelHydro = - massNgbs * ( POverRhoSquared     /localParticle->getOmega() * gradW
                                + POverRhoSquaredNgbs/activeParticle->getOmega() * gradWNgbs ) ;

      localParticle->setV( localParticle-> getV() + 0.0001 * accelHydro );

      assertion9( std::isfinite(accelHydro(0))         ,      localParticle->toString(), accelHydro, activeParticle->getDensity(), POverRhoSquared, POverRhoSquaredNgbs, dist, veldotdist, qAV,qAVNgbs );
      assertion9( std::isfinite(accelHydro(1))         ,      localParticle->toString(), accelHydro, activeParticle->getDensity(), POverRhoSquared, POverRhoSquaredNgbs, dist, veldotdist, qAV,qAVNgbs );
      assertion9( std::isfinite(localParticle->getV(0)),      localParticle->toString(), accelHydro, activeParticle->getDensity(), POverRhoSquared, POverRhoSquaredNgbs, dist, veldotdist, qAV,qAVNgbs );
      assertion9( std::isfinite(localParticle->getV(1)),      localParticle->toString(), accelHydro, activeParticle->getDensity(), POverRhoSquared, POverRhoSquaredNgbs, dist, veldotdist, qAV,qAVNgbs );
      assertion9( not tarch::la::greater(std::abs(localParticle ->getV(0)),1e+10), localParticle->toString(), accelHydro, activeParticle->getDensity(), POverRhoSquared, POverRhoSquaredNgbs, dist, veldotdist, qAV,qAVNgbs );
      assertion9( not tarch::la::greater(std::abs(localParticle ->getV(1)),1e+10), localParticle->toString(), accelHydro, activeParticle->getDensity(), POverRhoSquared, POverRhoSquaredNgbs, dist, veldotdist, qAV,qAVNgbs );
      assertion9( not tarch::la::greater(std::abs(activeParticle->getV(0)),1e+10), localParticle->toString(), accelHydro, activeParticle->getDensity(), POverRhoSquared, POverRhoSquaredNgbs, dist, veldotdist, qAV,qAVNgbs );
      assertion9( not tarch::la::greater(std::abs(activeParticle->getV(1)),1e+10), localParticle->toString(), accelHydro, activeParticle->getDensity(), POverRhoSquared, POverRhoSquaredNgbs, dist, veldotdist, qAV,qAVNgbs );
      assertion9( not tarch::la::greater(std::abs(accelHydro(0)),1e+10)          , localParticle->toString(), accelHydro, activeParticle->getDensity(), POverRhoSquared, POverRhoSquaredNgbs, dist, veldotdist, qAV,qAVNgbs );
      assertion9( not tarch::la::greater(std::abs(accelHydro(1)),1e+10)          , localParticle->toString(), accelHydro, activeParticle->getDensity(), POverRhoSquared, POverRhoSquaredNgbs, dist, veldotdist, qAV,qAVNgbs );


      }
  }
  
  // Add gravity:
  // Here we calculate the gravitational interaction among particles, i.e. the N-body system. 
  // Note that we only loop over local + active sets, not the N^2 particles, hence we neglect 
  // long-range contributions.
  // To avoid the 1/r^2 divergence due to particles being too close, we use a softening length parameter.
  
  const double SoftLen = 0.02;
  const double G_Newton= 1E-1;  // We can regulate the strength of gravity. For physical results we should use correct units.

  if ( tarch::la::greater( dist,0.0 ) ) {
    const double massLoc = localParticle ->getMass();
    const double massAct = activeParticle->getMass();
    
    double forcePart = -G_Newton*massLoc*massAct/ std::pow( dist*dist+SoftLen*SoftLen,1.5 );
    double accelGrav = forcePart/massLoc;
    
    // Update velocity using gravitational acceleration
    
    assertion3( std::isfinite(localParticle->getV(0)), localParticle->toString(), accelGrav, distVec );
    assertion3( std::isfinite(localParticle->getV(1)), localParticle->toString(), accelGrav, distVec );
    
//    localParticle->setV( localParticle->getV() + 0.0001 * accelGrav*distVec );
    
  }
""",
  touch_vertex_first_time_compute_particle_update_kernel="""
localParticle->setMoveState( globaldata::Particle::MoveState::NotMovedYet );
""",
  touch_vertex_last_time_compute_particle_update_kernel="""
  const double timeStepSize = 0.0001;
  if (
    localParticle->getMoveState()!=globaldata::Particle::MoveState::Moved
    and
    localParticle->getParallelState()==globaldata::Particle::ParallelState::Local
  ) {
  
    // Update position
    
    assertion1(  std::isfinite(localParticle->getV(0)), localParticle->toString() );
    assertion1(  std::isfinite(localParticle->getV(1)), localParticle->toString() );
    
    localParticle->setX( localParticle->getX() + timeStepSize * localParticle->getV() );
    
    // Boundary conditions
    for (int d=0; d<Dimensions; d++) {
      if ( localParticle->getX()(d)<0.0 ) {
        localParticle->setV( d, std::abs(localParticle->getV()(d)) );
//        localParticle->setX( d, 1.0 - std::abs(localParticle->getX()(d)) );

      }
      if ( localParticle->getX()(d)>1.0 ) {
        localParticle->setV( d,-std::abs(localParticle->getV()(d)) );
//        localParticle->setX( d,-1.0 + std::abs(localParticle->getX()(d)) );
      }
    }
    localParticle->setMoveState( globaldata::Particle::MoveState::Moved );
  }
  
  assertion2( localParticle->getDebugNeg()==-1.0, localParticle->toString(), localParticle->getDebugNeg() );


""",
  additional_includes="""
#include "Constants.h"
#include <cmath>
"""
  ))
move_particles.add_action_set( peano4.toolbox.particles.ParticleAMR(particles,particle_tree_analysis,min_particles_per_cell=10) )
move_particles.add_action_set( particle_tree_analysis )
project.solversteps.add_step(move_particles)

### Plots setup

I will have to plot the outcome (or any snapshot). With particles, each of my dumps produces two types of files: particle data and mesh data. For the mesh, I rely on the toolbox's grid plotting. The plot is the step where we don't need the AMR criterion - the particles don't move here after all. We however run the tree analysis, so it is up-to-date for the next step.

In [ ]:
# Particle plotter
particle_plotter = peano4.toolbox.particles.PlotParticlesInVTKFormat( "particles", particles, time_stamp_evaluation=peano4.toolbox.particles.PlotParticlesInVTKFormat.CountTimeSteps )
particle_plotter.add_attribute_to_plot(v_attr       , 2)
particle_plotter.add_attribute_to_plot(density_attr , 1)
particle_plotter.add_attribute_to_plot(pressure_attr, 1)
particle_plotter.add_attribute_to_plot(u_attr       , 1)
particle_plotter.add_attribute_to_plot(mass_attr    , 1)
particle_plotter.add_attribute_to_plot(smoothL_attr , 1)
particle_plotter.add_attribute_to_plot(smoothL_prev_attr , 1)
particle_plotter.add_attribute_to_plot(Omega_attr ,  1)
particle_plotter.add_attribute_to_plot(rhoDot_attr,  1)
particle_plotter.add_attribute_to_plot(DebugNeg_attr,1)

# Solution plotter
print_solution = peano4.solversteps.Step( "Plot", add_user_defined_actions=False )
print_solution.use_vertex(particles)
print_solution.use_cell(particle_tree_analysis.cell_marker)
print_solution.add_action_set( particle_tree_analysis )
print_solution.add_action_set( peano4.toolbox.PlotGridInPeanoBlockFormat( filename="grid", time_stamp_evaluation=peano4.toolbox.PlotGridInPeanoBlockFormat.CountTimeSteps ) )
print_solution.add_action_set( particle_plotter )

project.solversteps.add_step(print_solution)

## Generate the actual C++ code

Next, we generate the actual Peano 4 C++ code:

In [ ]:
project.generate()

## Write the main

Do not forget to include the generated header files, e.g., 
    
    //
    // =================
    // inserted manually
    // =================
    //
    #include "observers/CreateGrid.h"
    #include "observers/Init.h"
    #include "observers/Plot.h"
    #include "observers/Move.h"


## Build the code

In [ ]:
project.build()

## Run code

In [ ]:
# remove output files from previous runs
output_files = [ f for f in os.listdir(".") if f.endswith(".peano-patch-file") or f.endswith(".vtu") or f.endswith(".pvd") ]
for f in output_files:
  os.remove(f)

# success = project.run( args=["--threads", "1"], prefix=["mpirun", "-n", "1"] )
!./peano4

# Summary of SPH equations
We follow the review paper by Price et al (2018).

Notice that, throughout this, $h$ denotes the smoothing length.

#### SPH basics

\begin{align}
\rho_a& = \sum_b m_b W(|{\bf r}_a - {\bf r}_b|, h_a)\\
\delta\Omega_a\equiv\Omega_a &= 1 - \frac{\partial h_a}{\partial \rho_a}\sum_b m_b\frac{\partial W_{ab}(h_a)}{\partial h_a} \qquad\qquad (\delta\Omega_a=0 \quad \textrm{for constant $h$})\\
\frac{{\rm d}{\rho_a}}{{\rm d}t} &= \frac{1}{\Omega_a}\sum_b m_b{v}_{ab}\cdot{\bf \nabla}_aW_{ab}(h_a)
\end{align}

#### Equation of State (EoS) et al

\begin{align}
P &=(\gamma-1)\rho u \qquad \textrm{Equation of State (EoS)}
\end{align}



#### Euler equation (w/o artificial viscosity terms)

\begin{align}
\frac{{\rm d}{\bf v}_a}{{\rm d}t} &=-\sum_b m_b\left[\frac{P_a}{\rho_a^2\Omega_a}{\bf \nabla}_aW_{ab}(h_a) + \frac{P_b}{\rho_b^2\Omega_b}{\bf \nabla}_aW_{ab}(h_b) \right] + {\bf a}^{\rm self-grav}_a + {\bf a}^{\rm others}_a\\
\end{align}


#### Evolution of internal energy:

\begin{align}
\frac{{\rm d}u_a}{{\rm d}t} &=\frac{P_a}{\rho_a^2\Omega_a}\sum_b m_b{v}_{ab}\cdot{\bf \nabla}_aW_{ab}(h_a)\\
\end{align}


#### Newton-Raphson calculation of $\rho$ and (variable) $h$:

\begin{align}
\zeta(h_a)&\equiv m_a\left(\frac{\eta}{h_a}\right)^{3}-\sum_b m_b W_{ab}(h_a)=0\\
\textrm{Initial guess:} &\qquad\qquad h^{\rm ini}_a =h_a + \Delta t \frac{\partial h_a}{\partial\rho_a}\frac{d\rho_a}{dt}\\ 
\textrm{N-R iteration:}&\qquad\qquad h_{a,{\rm new}}=h_a - \frac{\zeta({h_a})}{\zeta'(h_a)} \qquad \textrm{where:}\qquad \zeta'(h_a)=-\frac{3\rho_a \Omega_a}{h_a} \\
\end{align}
 

#### Useful relations

\begin{align}
\nabla_aW_{ab}&=\frac{C_{\rm norm}}{h^4}f'(q)\hat{{\bf r}}_{ab} \\
\frac{\partial W_{ab}}{\partial h}(r,h)&=-\frac{C_{\rm norm}}{h^4}[3f(q)+qf'(q)] \\
\frac{\partial h_a}{\partial \rho_a} &= -\frac{3h_a}{\rho_a} \\
\end{align}

## 